# TATR-Span Ablation Study (E0~E3)

**실험 구성**
| ID | 설명 | 핵심 옵션 |
|----|----|----|
| E0 | Baseline TATR (원본 재현) | `--no_span_branch --no_grid_snapping` |
| E1 | + Span Attribute Branch | ordinal regression head (K=8) |
| E2 | + Hard Grid-Snapping | curriculum warmup 5 epochs |
| E3 | + Soft Grid-Snapping | curriculum warmup 5 epochs |

3 seeds × 4 실험 = 12회 학습, 각 20 epochs

---
> **런타임 설정**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

## 0. GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 1. 환경 설정

In [ ]:
# 필수 패키지 설치
!pip install -q pycocotools
print("Done")

In [ ]:
import os

REPO_DIR = "/content/t1"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Jax0303/t1.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Already cloned — pulled latest")

%cd {REPO_DIR}
!git log --oneline -3

## 2. Google Drive 마운트 & 데이터 경로 설정

**Drive 업로드 필요 파일 (D드라이브 → Drive):**
```
MyDrive/PubTables-1M-Structure/
  images/          ← 94,959개 jpg (21GB, 필수)
  train/           ← 86,284개 xml
  val/             ← xml
  test/            ← xml
  train_filelist.txt
  val_filelist.txt
  test_filelist.txt
```

> **Tip**: 이미지 전체(21GB) 업로드가 오래 걸리면 아래 `SUBSET_MODE = True`로 소규모 먼저 테스트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────
DATA_ROOT  = "/content/drive/MyDrive/PubTables-1M-Structure"  # Drive 경로
OUTPUT_DIR = "/content/outputs/ablation"                       # Colab 로컬 저장

# 서브셋 모드: True → 빠른 smoke test (train 2000, val 300)
# False → 전체 데이터 학습 (T4 기준 실험 1회 약 3~4시간)
SUBSET_MODE = True
TRAIN_MAX   = 2000 if SUBSET_MODE else None
VAL_MAX     = 300  if SUBSET_MODE else None
EPOCHS      = 5    if SUBSET_MODE else 20

# ── 검증 ───────────────────────────────────────────────────
import os
for d in [DATA_ROOT, os.path.join(DATA_ROOT, 'images'),
          os.path.join(DATA_ROOT, 'train')]:
    exists = os.path.exists(d)
    count  = len(os.listdir(d)) if exists else 0
    print(f"{'✅' if exists else '❌'} {d}  ({count} files)")

## 3. Sanity Check (CPU, 데이터 불필요)

In [ ]:
!python sanity_check.py

## 4. 학습 실행 (E0 ~ E3)

12회 전체 실행 (`run_all_colab()`) 또는 실험 단위로 개별 실행 가능.

In [ ]:
import subprocess, sys, os, time

CONFIG      = "tatr_base/src/structure_config.json"
TRAIN_SCRIPT= "tatr_base/src/main.py"
BACKBONE    = "resnet18"
SEEDS       = [42, 43, 44]

def run_one(exp_id, seed, extra_args=(), epochs=EPOCHS,
            train_max=TRAIN_MAX, val_max=VAL_MAX):
    out_dir  = f"{OUTPUT_DIR}/{exp_id}/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)
    log_path = f"{out_dir}/run.log"

    cmd = [
        sys.executable, TRAIN_SCRIPT,
        "--data_root_dir",  DATA_ROOT,
        "--config_file",    CONFIG,
        "--backbone",       BACKBONE,
        "--data_type",      "structure",
        "--mode",           "train",
        "--epochs",         str(epochs),
        "--model_save_dir", out_dir,
        "--metrics_save_filepath", f"{out_dir}/metrics.json",
        "--device",         "cuda",
        "--seed",           str(seed),
        "--num_workers",    "2",
    ]
    if train_max: cmd += ["--train_max_size", str(train_max)]
    if val_max:   cmd += ["--val_max_size",   str(val_max)]
    cmd += list(extra_args)

    print(f"\n{'='*60}")
    print(f"  {exp_id}  seed={seed}  epochs={epochs}")
    print(f"  subset={'ON' if SUBSET_MODE else 'OFF'}  out={out_dir}")
    print(f"{'='*60}")

    t0 = time.time()
    with open(log_path, 'w') as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end='')   # Colab 실시간 출력
            log.write(line)
        proc.wait()

    elapsed = (time.time() - t0) / 60
    status  = '✅' if proc.returncode == 0 else '❌'
    print(f"\n{status} {exp_id} seed={seed} — {elapsed:.1f} min  (rc={proc.returncode})")
    return proc.returncode == 0

In [ ]:
# ── E0: Baseline TATR (span branch 없음, grid-snapping 없음) ──
for seed in SEEDS:
    run_one("E0_baseline", seed, extra_args=(
        "--no_span_branch",
        "--no_grid_snapping",
    ))

In [ ]:
# ── E1: + Span Attribute Branch (ordinal regression, grid-snap 없음) ──
for seed in SEEDS:
    run_one("E1_span", seed, extra_args=(
        "--span_loss_coef", "0.5",
        "--no_grid_snapping",
    ))

In [ ]:
# ── E2: + Hard Grid-Snapping (curriculum warmup 5 epochs) ──
for seed in SEEDS:
    run_one("E2_snap_hard", seed, extra_args=(
        "--span_loss_coef", "0.5",
        "--grid_snapping",  "hard",
        "--n_warm",         "5",
    ))

In [ ]:
# ── E3: + Soft Grid-Snapping (curriculum warmup 5 epochs) ──
for seed in SEEDS:
    run_one("E3_snap_soft", seed, extra_args=(
        "--span_loss_coef", "0.5",
        "--grid_snapping",  "soft",
        "--n_warm",         "5",
    ))

## 5. 복잡도별 평가 (eval_by_complexity)

In [ ]:
!python tatr_base/src/eval_by_complexity.py \
    --results_dir  {OUTPUT_DIR} \
    --data_root    {DATA_ROOT} \
    --xml_subdir   test \
    --output_csv   {OUTPUT_DIR}/results_by_complexity.csv

## 6. 결과 집계 (Table A / Table B)

In [ ]:
!python aggregate_results.py \
    --results_dir {OUTPUT_DIR} \
    --output_csv  {OUTPUT_DIR}/summary_results.csv

## 7. 결과 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

csv_path = f"{OUTPUT_DIR}/results_by_complexity.csv"

try:
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows")
    display(df[df['split'] == 'all'][['experiment_id', 'seed', 'GriTS_Top', 'GriTS_Loc', 'GriTS_Con']])
except FileNotFoundError:
    print(f"CSV not found: {csv_path}\n→ Section 5 (eval_by_complexity) 를 먼저 실행하세요")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

csv_path = f"{OUTPUT_DIR}/results_by_complexity.csv"
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    print("CSV 없음 — Section 5 먼저 실행"); raise SystemExit

# ── Simple vs Complex GriTS_Loc 비교 ──
exps = ['E0_baseline', 'E1_span', 'E2_snap_hard', 'E3_snap_soft']
splits = ['simple', 'complex']
colors = {'simple': '#4C72B0', 'complex': '#DD8452'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, metric in zip(axes, ['GriTS_Loc', 'GriTS_Top']):
    for split in splits:
        sub = df[df['split'] == split].copy()
        sub[metric] = sub[metric].astype(float)
        means = [sub[sub['experiment_id'] == e][metric].mean() for e in exps]
        stds  = [sub[sub['experiment_id'] == e][metric].std() for e in exps]
        x = np.arange(len(exps))
        offset = -0.2 if split == 'simple' else 0.2
        ax.bar(x + offset, means, 0.35, yerr=stds, label=split,
               color=colors[split], capsize=4, alpha=0.85)
    ax.set_title(metric, fontsize=13)
    ax.set_xticks(np.arange(len(exps)))
    ax.set_xticklabels(['E0\nBaseline', 'E1\nSpan', 'E2\nHard', 'E3\nSoft'], fontsize=10)
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Ablation: Simple vs Complex Tables (mean ± std, 3 seeds)', fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ablation_main.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {OUTPUT_DIR}/ablation_main.png")

In [ ]:
# ── k_bin breakdown: E0 vs E3 ──
import pandas as pd, matplotlib.pyplot as plt, numpy as np

csv_path = f"{OUTPUT_DIR}/results_by_complexity.csv"
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    print("CSV 없음"); raise SystemExit

kbins = ['k=1', 'k=2', 'k=3~4', 'k>=5']
e0_means, e3_means = [], []

for kb in kbins:
    sub0 = df[(df['experiment_id'] == 'E0_baseline') & (df['k_bin'] == kb)]
    sub3 = df[(df['experiment_id'] == 'E3_snap_soft') & (df['k_bin'] == kb)]
    e0_means.append(sub0['GriTS_Loc'].astype(float).mean() if len(sub0) else float('nan'))
    e3_means.append(sub3['GriTS_Loc'].astype(float).mean() if len(sub3) else float('nan'))

x = np.arange(len(kbins))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, e0_means, 0.35, label='E0 Baseline', color='#4C72B0', alpha=0.85)
ax.bar(x + 0.2, e3_means, 0.35, label='E3 Soft-Snap', color='#55A868', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(kbins)
ax.set_ylabel('GriTS_Loc')
ax.set_ylim(0, 1)
ax.set_title('GriTS_Loc by Span Complexity: E0 vs E3')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ablation_kbin.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {OUTPUT_DIR}/ablation_kbin.png")

## 8. 결과 Drive로 백업

In [ ]:
import shutil, os, time

timestamp  = time.strftime('%Y%m%d_%H%M')
backup_dst = f"/content/drive/MyDrive/ablation_results_{timestamp}"

shutil.copytree(OUTPUT_DIR, backup_dst)
print(f"Backed up → {backup_dst}")

# 백업된 파일 목록
for root, dirs, files in os.walk(backup_dst):
    level = root.replace(backup_dst, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files:
            print(f"{indent}  {f}")

## 9. 개별 실험 재실행 / 체크포인트 이어받기

Colab 세션이 끊겼을 때 특정 실험만 다시 돌리는 방법.

In [ ]:
# 예시: E3 seed=42 만 다시 실행 (체크포인트 이어받기)
RESUME_EXP  = "E3_snap_soft"
RESUME_SEED = 42
LOAD_PATH   = f"{OUTPUT_DIR}/{RESUME_EXP}/seed{RESUME_SEED}/model.pth"  # 마지막 저장 모델

if os.path.exists(LOAD_PATH):
    run_one(RESUME_EXP, RESUME_SEED,
            extra_args=(
                "--span_loss_coef", "0.5",
                "--grid_snapping",  "soft",
                "--n_warm",         "5",
                "--model_load_path", LOAD_PATH,
            ))
else:
    print(f"체크포인트 없음: {LOAD_PATH}")
    print("→ Section 4의 해당 셀을 처음부터 실행하세요")

## 부록. 서브셋 → 전체 전환

```python
# Section 2의 설정 변경
SUBSET_MODE = False   # ← 이걸 바꾸면
TRAIN_MAX   = None    # 전체 86,284개 사용
VAL_MAX     = None
EPOCHS      = 20      # 풀 20 epochs
```

T4 기준 예상 시간:
- 서브셋(2000샘플, 5 epochs): 실험 1회 약 **15~20분**
- 전체(86K샘플, 20 epochs): 실험 1회 약 **3~4시간**
- 전체 12회: **36~48시간** → Colab Pro+ 권장 (A100)

> Colab 무료 T4는 세션당 최대 12시간. 실험을 E0→E3 순서로 나눠서 돌리고 Drive로 백업.